# MiniCPM5-1B fine-tune on Modal GPU Notebook

Interactive path for the **Modal** + **Well-Tuned** hackathon tracks.

Model: [openbmb/MiniCPM5-1B](https://huggingface.co/openbmb/MiniCPM5-1B)

For reproducible sweeps, use `modal run research/modal/finetune_app.py` instead.

In [ ]:
# Verify GPU (Modal Notebooks provide CUDA)
!nvidia-smi

In [ ]:
# Clone hackathon repo (replace with your fork URL if needed)
!git clone https://github.com/YOUR_USER/small-model-hackathon.git /root/repo || true
%cd /root/repo

In [ ]:
# Install finetune + lm-eval deps
!pip install -q uv
!uv sync --frozen --package inference --package slm-evals --group finetune --group lm-eval --no-dev

In [ ]:
import os
os.environ["TRUST_REMOTE_CODE"] = "true"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

## Smoke fine-tune (LoRA, 20 steps)

Uses the lesson-agent chat dataset by default.

In [ ]:
!uv run python research/finetune.py \
  --preset minicpm5-1b \
  --mode lora \
  --dataset research/data/education-lesson-chat.jsonl \
  --format chat \
  --out ./models/finetuned/minicpm5-1b-lesson-lora \
  --max_steps 20

## Baseline lm-eval (smoke)

In [ ]:
!uv run --package slm-evals slm-lm-eval \
  --config research/evals/configs/lm_eval_smoke.yaml \
  --preset minicpm5-1b \
  --experiment-name minicpm5-1b__notebook-baseline

## Post-train lm-eval (adapter)

In [ ]:
!uv run --package slm-evals slm-lm-eval \
  --config research/evals/configs/lm_eval_smoke.yaml \
  --model openbmb/MiniCPM5-1B \
  --adapter ./models/finetuned/minicpm5-1b-lesson-lora \
  --experiment-name minicpm5-1b-lesson-lora__notebook \
  --compare-to results/lm_eval/minicpm5-1b__notebook-baseline/results.json

## Sample generation

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "openbmb/MiniCPM5-1B"
adapter_dir = "./models/finetuned/minicpm5-1b-lesson-lora"

tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

prompt = "Explain photosynthesis in one short paragraph for a 10-year-old."
if tokenizer.chat_template:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    text = prompt

ids = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))